In [2]:
import numpy as np
import json
import random
from collections import deque
from pathlib import Path

In [3]:
NUM_GRIDS = 100

random.seed(42)
np.random.seed(42)

In [4]:
def is_goal_reachable(obstacles, start, goal, grid_size):
    queue = deque([start])
    visited = {start}

    directions = [
        (-1, 0),  # North
        (1, 0),   # South
        (0, -1),  # West
        (0, 1)    # East
    ]

    while queue:
        r, c = queue.popleft()

        if (r, c) == goal:
            return True

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            next_pos = (nr, nc)

            if (
                0 <= nr < grid_size
                and 0 <= nc < grid_size
                and next_pos not in obstacles
                and next_pos not in visited
            ):
                visited.add(next_pos)
                queue.append(next_pos)

    return False


In [5]:
def generate_grids(num_grids, grid_size, obstacle_range=(4, 6)):
    grids = []
    seen = set()

    while len(grids) < num_grids:
        start_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))
        goal_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))
        while goal_pos == start_pos:
            goal_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))

        
        open_positions = [
            (r, c)
            for r in range(grid_size)
            for c in range(grid_size)
            if (r, c) not in [start_pos, goal_pos]
        ]



        num_obstacles = random.randint(obstacle_range[0], obstacle_range[1])
        obstacles = frozenset(
            random.sample(open_positions, num_obstacles)
        )

        # Unique obstacle configuration
        if obstacles in seen:
            continue

        # Only retain grids with reachable goals
        if is_goal_reachable(obstacles, start_pos, goal_pos, grid_size):
          seen.add(obstacles)

          grid = np.full((grid_size, grid_size), '.')

          grid[start_pos] = 'S'  # Start
          grid[goal_pos] = 'G'  # Goal

          for r, c in obstacles:
              grid[r, c] = 'B'

          grids.append({
            'id': len(grids),
            'grid': grid.tolist(),
            'start': start_pos,
            'goal': goal_pos,
            'obstacles': list(obstacles)
          })

    return grids

In [6]:
def save_grids_to_json(grids, filename):
    print("Current directory:", Path.cwd())

    with open(f"data/{filename}", "w") as f:
        json.dump(grids, f, indent=2)

In [ ]:
grids_5 = generate_grids(num_grids=NUM_GRIDS, grid_size=5, obstacle_range=(4, 6))

print("Number of grids (5x5):", len(grids_5))

save_grids_to_json(grids_5, "grids_5.json")



Number of grids (5x5): 100
Current directory: /home/sm/projects/llm-grid-solver


In [8]:
grids_7 = generate_grids(num_grids=NUM_GRIDS, grid_size=7, obstacle_range=(6, 8))

print("Number of grids (7x7):", len(grids_7))

save_grids_to_json(grids_7, "grids_7.json")

Number of grids (7x7): 100
Current directory: /home/sm/projects/llm-grid-solver


In [9]:
grids_10 = generate_grids(num_grids=NUM_GRIDS, grid_size=10, obstacle_range=(10, 12))

print("Number of grids (10x10):", len(grids_10))

save_grids_to_json(grids_10, "grids_10.json")

Number of grids (10x10): 100
Current directory: /home/sm/projects/llm-grid-solver


In [11]:
print("Grid 5x5 samples:")

for grid in grids_5[:2]:
    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid['grid'])}")

Grid 5x5 samples:
Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .


In [12]:
print("Grid 7x7 samples:")

for grid in grids_7[:2]:
    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid['grid'])}")

Grid 7x7 samples:
Grid:
. . . . . . .
. . . . . . .
. . . . . . B
B B . B G . .
. . B . . . .
B B . . . . .
. S . . . . .
Grid:
. . B . . . .
. . . . . . .
B . . . . . B
. . . B . B .
. . B . . . .
. G . . . . B
. S . . . . B


In [13]:
print("Grid 10x10 samples:")

for grid in grids_10[:2]:
    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid['grid'])}")

Grid 10x10 samples:
Grid:
. . . . . . . G B .
B . . . . . . . . .
. . . . . . . . . .
. . B . . . . . B B
. . . . . . . . . .
. . B . . . . . . B
. . B . . . . . . S
. B . . . . . . . .
. . . . B . . . . B
B . . . . . . . . .
Grid:
. . . . . . . . . .
. . . . . . B . B .
. . . . . . B S . .
. . . . . . . . . .
. B . . B B . . . .
. . B . . . . . . .
. . . . . . . G . .
. . . . . . . . . B
. . . . . . . . . .
. . . B . B . . . B
